# Monte Carlo, explained from scratch

This notebook assumes **you have never written Python**. Every line of code is explained,
including what the punctuation means.

Run it with the project environment: `/Users/mac/miniconda3/envs/ml_env/bin/jupyter-lab`,
and pick the kernel named `ml_env` in the top right.

---

## What are we actually trying to do?

We have a machine that predicts two numbers about a crystal:

- **K**, the *bulk modulus* — how hard it is to squeeze the crystal
- **G**, the *shear modulus* — how hard it is to twist it

From those two numbers, some physics formulas give us **κ (kappa)** — how well heat flows
through the crystal. That is the number we care about.

Here is the problem. Our machine is not certain about K and G. It says something like
*"K is about 54, give or take 2"*. So κ cannot be certain either.

**The question: if K and G are each a bit uncertain, how uncertain is κ?**

---

## The idea behind Monte Carlo, with no maths

Imagine a shop where you buy two ingredients, and the price of a cake depends on both in a
complicated way. The shopkeeper tells you:

> "Flour costs about £3, give or take 20p. Sugar costs about £2, give or take 40p."

You want to know: **what might the cake cost?**

One approach is calculus — work out how sensitive the cake price is to each ingredient and
combine them. That works if the relationship is simple. Ours is not.

The Monte Carlo approach is much more direct, almost childishly so:

1. **Guess** a flour price and a sugar price, at random, but respecting the "give or take"
2. **Work out** the cake price for that guess
3. **Do that 2000 times**, writing down all 2000 cake prices
4. **Look at the list.** If 90% of the prices landed between £8 and £12, that is your answer

That is the entire method. No calculus. Just doing the sum many times with slightly
different inputs and looking at the spread of answers.

"Monte Carlo" is only a nickname — after the casino, because the method relies on random
numbers.

---
# Part 1 · Getting set up

The first code cell does no maths at all. It only loads the tools we need.

**A few things about Python before we start:**

- A line starting with `#` is a **comment**. Python ignores it completely. It is a note for
  humans. You will see one above nearly every line below.
- `import X` means *"load the toolbox called X so I can use its tools"*.
- `import X as Y` means *"load toolbox X, but let me refer to it by the shorter name Y"*.
- The `=` sign does **not** mean "equals" like in maths. It means **"store this, and call it
  by this name"**. So `age = 30` means "remember the number 30 under the name `age`".

In [ ]:
# Load the toolbox that does heavy number-crunching. It MUST be loaded before
# the others on this computer, or the notebook crashes. (The reason is a clash
# between two copies of a background maths library - not something you need to
# understand, just something to keep in this order.)
import torch

# "os" and "sys" are toolboxes for talking to the computer itself - finding
# folders, and telling Python where to look for our project's own code.
import os
import sys

# "math" has ordinary calculator functions, like square root and logarithm.
import math

# "numpy" is the toolbox for working with LISTS of numbers all at once.
# We nickname it "np" because we type it a lot. Almost every line below uses it.
import numpy as np

# "pandas" is the toolbox for reading spreadsheets (tables of data).
# We nickname it "pd".
import pandas as pd

# "matplotlib.pyplot" draws graphs. Nicknamed "plt".
import matplotlib.pyplot as plt

# Now a message so we can see the cell finished. print() puts text on the screen.
print("All toolboxes loaded.")

### Finding the project's own code

Our project has its own physics formulas written in a file. Rather than copying them here
(where they could drift out of date), we load that file and use it directly.

**More Python:**

- `os.getcwd()` asks the computer *"which folder am I in right now?"*
- A **function** is a named piece of code you can run. You run it by writing its name
  followed by round brackets: `print("hello")` runs the function `print` and hands it the
  text `"hello"`.
- The things you hand a function go inside the brackets and are called **arguments**.

In [ ]:
# Ask the computer which folder this notebook is sitting in.
current_folder = os.getcwd()

# This notebook lives in a folder called "notebooks", which sits INSIDE the
# project folder. So the project folder is one level up.
# os.path.basename(...) gives just the last part of a folder path.
# The word "if" means: only do the next bit when the test is true.
if os.path.basename(current_folder) == "notebooks":
    PROJECT = os.path.dirname(current_folder)   # go up one level
else:
    PROJECT = current_folder                    # we are already at the top

# Tell Python it is allowed to load code from these two folders.
sys.path.insert(0, PROJECT)
sys.path.insert(0, os.path.join(PROJECT, "scripts", "cgcnn"))

# Our physics file is named "07_predict_kappa.py". Because that name starts with
# a digit, Python's normal "import" cannot load it. This is a workaround that
# loads a file by giving its name as text instead.
from importlib import import_module
physics_file = import_module("07_predict_kappa")

# Pull out the one function we care about and give it a short name.
# From here on, "slack_physics" IS the project's real physics.
slack_physics = physics_file.slack_physics

print("Loaded the real physics from:")
print("   ", physics_file.__file__)

---
# Part 2 · What the machine tells us

Our predictions live in a spreadsheet file. Let us open it and look.

**More Python:**

- `pd.read_csv("somefile.csv")` opens a spreadsheet and hands back a table.
- A table has **columns** with names. `table["K_VRH_pred"]` means *"give me the column
  called K_VRH_pred"*. The square brackets mean **"pick out this bit"**.
- `.head(5)` means *"just the first 5 rows, please"*.

In [ ]:
# Build the full address of the spreadsheet file.
# os.path.join glues folder names together with the right kind of slash.
predictions_file = os.path.join(PROJECT, "results", "cgcnn", "pink_moduli_predictions.csv")

# Open it. "predictions" is now a table sitting in the computer's memory.
predictions = pd.read_csv(predictions_file)

# len(...) counts how many rows the table has.
print("The table has", len(predictions), "rows - one per crystal.")
print()

# Choose four columns to look at, and show the first five rows.
# The inner square brackets [ ... ] make a LIST of column names.
print(predictions[["formula", "K_VRH_pred", "K_VRH_spread_log10", "G_VRH_pred"]].head(5))

### What those columns mean

| column | meaning |
|---|---|
| `formula` | which crystal, e.g. `LaGa` is lanthanum-gallium |
| `K_VRH_pred` | the machine's **best guess** for K |
| `K_VRH_spread_log10` | the machine's **"give or take"** for K |
| `G_VRH_pred` | best guess for G |

### Where the "give or take" comes from

We did not train one machine. We trained **three**, each starting from a different random
setting. For any crystal they give three slightly different answers.

- their **average** is the best guess
- **how much they disagree** is the give-or-take

If all three agree closely, we are confident. If they disagree wildly, we are not. That
disagreement is the only uncertainty information we have, and it is what the whole Monte
Carlo is built on.

### One oddity: the give-or-take is a *multiplier*, not an amount

The column is called `..._spread_log10`. That means it is measured on a **logarithm** of the
modulus, not on the modulus itself.

In plain terms: the machine's error is a **percentage**, not a fixed number of units. Being
wrong by 12 units is a disaster for a soft crystal worth 20, and irrelevant for a hard one
worth 400. Saying "wrong by about 6%" means the same thing for both.

The next cell converts one of these log numbers into a percentage so you can see it.

In [ ]:
# .median() finds the middle value of a column - half are bigger, half smaller.
typical_spread = predictions["K_VRH_spread_log10"].median()

# Show it, rounded to 4 decimal places. The ":.4f" inside the curly braces is a
# formatting instruction meaning "show 4 digits after the point".
print(f"A typical give-or-take, in log units: {typical_spread:.4f}")

# To turn a log10 give-or-take into a percentage, raise 10 to that power.
# In Python, ** means "to the power of". So 10 ** 2 is 100.
multiplier = 10 ** typical_spread

print(f"10 to the power of that is: {multiplier:.4f}")
print(f"So the machine is typically uncertain by about {100 * (multiplier - 1):.1f}%")

---
# Part 3 · Doing it for ONE crystal, slowly

We will now follow a single crystal all the way through. This is the heart of the notebook.

**More Python:**

- `table.iloc[0]` means *"give me row number 0"*. Python counts from **0**, not 1, so row 0
  is the first row.
- Once you have a row, `row.formula` means *"the formula column of this row"*. The dot means
  **"belonging to"**.

In [ ]:
# Take the very first crystal in the table.
crystal = predictions.iloc[0]

# Read its four numbers out into plainly-named variables, so the rest of the
# notebook is readable.
name       = crystal.formula                # e.g. "LaGa"
K_guess    = crystal.K_VRH_pred             # best guess for K
K_giveTake = crystal.K_VRH_spread_log10     # give-or-take for K, in log units
G_guess    = crystal.G_VRH_pred             # best guess for G
G_giveTake = crystal.G_VRH_spread_log10     # give-or-take for G, in log units

print("We will follow this crystal:", name)
print(f"   K = {K_guess:.2f}, give or take {K_giveTake:.4f} in log units")
print(f"   G = {G_guess:.2f}, give or take {G_giveTake:.4f} in log units")

### The crystal's fixed facts

The physics also needs four more numbers: the volume of the crystal's box, its density, its
average atom weight, and how many atoms are in the box.

**These are not predictions.** They are read straight off the crystal's description file.
They are facts, so they are **not uncertain**, and we will not be randomising them. Only K
and G get randomised.

In [ ]:
# Open a second spreadsheet that holds these structural facts.
facts_file = os.path.join(PROJECT, "results", "cgcnn", "pink_kappa_predictions.csv")
facts_table = pd.read_csv(facts_file)

# Find the row for OUR crystal. Read this inside-out:
#   facts_table.material_id == crystal.material_id
#       -> asks, for every row, "is this the same crystal?" (true or false)
#   facts_table[ ... ]
#       -> keeps only the rows where the answer was true
#   .iloc[0]
#       -> takes the first (and only) surviving row
same_crystal = facts_table.material_id == crystal.material_id
facts = facts_table[same_crystal].iloc[0]

# float(...) makes sure each value is a plain decimal number.
volume  = float(facts["Volume (A3)"])         # size of the box
density = float(facts["Density (g cm-3)"])    # how heavy it is for its size
mass    = float(facts["Atomic mass (amu)"])   # average weight of one atom
n_atoms = float(facts["Number of Atoms"])     # how many atoms in the box

print("Fixed facts about", name, "- these never change:")
print(f"   volume  = {volume:.1f}")
print(f"   density = {density:.3f}")
print(f"   mass    = {mass:.1f}")
print(f"   atoms   = {n_atoms:.0f}")

## Step 1 of 3 — make 2000 guesses

Now the actual Monte Carlo begins.

We want 2000 plausible values of K. Not 2000 random numbers — 2000 numbers that respect
what the machine told us: *centred on the best guess, scattered by the give-or-take*.

The recipe for one guess is:

> **guess = best guess + (give-or-take × a random wobble)**

where the "random wobble" is a number drawn from the famous bell curve — usually near zero,
occasionally further out, rarely far out. That shape is what makes the guesses cluster near
the best guess instead of spreading evenly.

**More Python:**

- `np.random.RandomState(0)` creates a random-number machine. The `0` is a **seed**: giving
  the same seed always produces the same "random" numbers, so this notebook gives identical
  results every time you run it.
- `.standard_normal(2000)` asks that machine for 2000 bell-curve wobbles.

In [ ]:
# How many guesses to make. 2000 is what the real pipeline uses.
n_guesses = 2000

# Create the random-number machine, seeded with 0 so results never change.
dice = np.random.RandomState(0)

# Ask for 2000 bell-curve wobbles for K, and 2000 more for G.
# These are separate requests, so the K wobbles and G wobbles are unrelated.
wobbles_K = dice.standard_normal(n_guesses)
wobbles_G = dice.standard_normal(n_guesses)

# Look at what a wobble actually is.
print("The first five wobbles:", np.round(wobbles_K[:5], 3))
print(f"Their average is about {wobbles_K.mean():.3f} (should be near 0)")
print(f"Their spread is about  {wobbles_K.std():.3f} (should be near 1)")

Now apply the recipe. One important detail: because the give-or-take is measured in **log**
units, we must do the arithmetic in log units too, and only convert back at the end.

So it is really three small steps:

1. take the log of the best guess
2. add `give-or-take × wobble`
3. undo the log, to get back to a normal number

In [ ]:
# Step 1: take the logarithm of the best guess.
log_of_K_guess = np.log10(K_guess)

# Step 2: add the wobbles, scaled by the give-or-take.
# Because "wobbles_K" is a list of 2000 numbers, this one line does the addition
# 2000 times over - once for each wobble. That is what numpy is for: you write
# the sum once and it happens to every number in the list.
log_K_guesses = log_of_K_guess + K_giveTake * wobbles_K

# Step 3: undo the logarithm to get back to ordinary numbers.
K_guesses = 10 ** log_K_guesses

# Exactly the same three steps for G, written in one line this time.
G_guesses = 10 ** (np.log10(G_guess) + G_giveTake * wobbles_G)

print(f"Made {len(K_guesses)} guesses for K and {len(G_guesses)} for G.")
print()
print(f"The machine's single best guess for K was {K_guess:.2f}")
print(f"Our 2000 guesses run from {K_guesses.min():.2f} up to {K_guesses.max():.2f}")
print(f"and their middle value is {np.median(K_guesses):.2f} - right back at the best guess,")
print("which is exactly what we want: we scattered AROUND it, we did not move it.")

## Step 2 of 3 — work out κ for every guess

Now we hand each of the 2000 pairs to the physics and get 2000 answers back.

This is the step where Monte Carlo earns its keep. We are not approximating the physics or
simplifying it. We are **running the real thing, 2000 times**.

**More Python:**

- `slack_physics(a, b, c, d, e, f)` runs the physics function with six pieces of
  information handed to it, in that order.
- It hands back a **dictionary** — a labelled collection. `answer["kappa_cal"]` means *"the
  part of the answer labelled kappa_cal"*.

In [ ]:
# Run the physics. The first two arguments are our LISTS of 2000 guesses;
# the last four are the single fixed facts, which apply to every guess.
answer = slack_physics(K_guesses, G_guesses, volume, density, mass, n_atoms)

# Pull out the kappa values - one for each of our 2000 guesses.
kappa_guesses = answer["kappa_cal"]

print(f"We now have {len(kappa_guesses)} possible values of kappa.")
print(f"The smallest is {kappa_guesses.min():.3f}")
print(f"The largest  is {kappa_guesses.max():.3f}")
print()

# For comparison: what does the physics say if we DON'T randomise anything,
# and just use the single best guesses? That is the "point estimate".
best_answer = slack_physics(np.array([K_guess]), np.array([G_guess]),
                            volume, density, mass, n_atoms)
point_estimate = best_answer["kappa_cal"][0]
print(f"Using only the best guesses, kappa = {point_estimate:.3f}")
print("Our 2000 values are scattered around roughly that.")

## Step 3 of 3 — read the answer off the list

We have 2000 possible values of κ. What do we report?

We sort them and find the value that 5% fall below, and the value that 95% fall below.
Those two numbers bracket the middle 90% of our guesses. That is our uncertainty interval.

These are called **percentiles**. The 5th percentile is the value that 5% of the list sits
below.

**More Python:**

- `np.percentile(list, [5, 50, 95])` finds three percentiles at once and hands back three
  numbers.
- Writing `a, b, c = ...` on the left unpacks those three numbers into three names.

In [ ]:
# Find the 5th, 50th (middle) and 95th percentiles of our 2000 kappa values.
low, middle, high = np.percentile(kappa_guesses, [5, 50, 95])

print("Our answer for", name, ":")
print(f"   kappa is most likely about {middle:.3f}")
print(f"   and we are 90% confident it lies between {low:.3f} and {high:.3f}")
print()
print(f"That range is a factor of {high / low:.1f} wide - so this is not a precise answer,")
print("and pretending otherwise would be the mistake this whole exercise exists to avoid.")

### Why percentiles, and not "plus or minus"?

You might expect us to say "κ = 4.5, plus or minus 2". That would assume the uncertainty is
**symmetric** — the same amount above and below.

It is not. Check it:

In [ ]:
# Distance from the middle down to the low end.
distance_down = middle - low

# Distance from the middle up to the high end.
distance_up = high - middle

print(f"From the middle DOWN to the low end:  {distance_down:.3f}")
print(f"From the middle UP   to the high end: {distance_up:.3f}")
print()
print(f"The upper side is {distance_up / distance_down:.1f} times longer than the lower side.")
print()
print("So 'plus or minus one number' would be wrong. It would either overstate")
print("the low side or understate the high side. Percentiles just report what")
print("the 2000 answers actually did, and assume nothing about their shape.")

### A picture of the whole thing

Left: the 2000 guesses we fed in. Right: the 2000 answers we got out.

Notice the shape changes. We put in a symmetric bell curve; we got out something
lopsided, with a long tail to the right. **The physics did that.** It is precisely why we
could not have used a simple "plus or minus".

In [ ]:
# Make a drawing area with two panels side by side.
figure, panels = plt.subplots(1, 2, figsize=(12, 4))

# --- left panel: what went IN --------------------------------------------
left = panels[0]
# A histogram sorts the numbers into bins and draws a bar for each bin,
# showing how many landed there.
left.hist(K_guesses, bins=50, color="#1450AA", edgecolor="white")
# Draw a vertical line at the original best guess.
left.axvline(K_guess, color="#D97B12", linewidth=2, label="the best guess")
left.set_xlabel("our 2000 guesses for K")
left.set_ylabel("how many landed here")
left.set_title("IN: a symmetric bell curve")
left.legend()

# --- right panel: what came OUT ------------------------------------------
right = panels[1]
right.hist(kappa_guesses, bins=50, color="#7A3FA0", edgecolor="white")
# Mark our three reported numbers.
right.axvline(low,    color="#D97B12", linewidth=2, linestyle="--", label="5% below here")
right.axvline(middle, color="#1F2937", linewidth=2,                 label="middle")
right.axvline(high,   color="#D97B12", linewidth=2, linestyle="--", label="95% below here")
right.set_xlabel("the 2000 resulting values of kappa")
right.set_title("OUT: lopsided, with a long right tail")
right.legend()

plt.tight_layout()   # tidy up the spacing
plt.show()           # actually draw it

---
# Part 4 · Why we work in logarithms

Earlier we said the give-or-take is a percentage, so we do the arithmetic in log units.
Here is the concrete reason that matters, and it is not a technicality.

**A modulus cannot be negative.** A crystal cannot be *less than infinitely* squeezable.
Negative would be meaningless, and the physics formula takes a **square root** of it — and
you cannot take the square root of a negative number.

Working in logs makes a negative guess **impossible**, because raising 10 to any power
always gives something above zero.

If we instead added the wobble directly to the modulus, a big enough wobble on a soft
crystal could push it below zero. Let us prove that happens on real data.

In [ ]:
# Find the SOFTEST crystal in the whole table - the one with the smallest K.
# .idxmin() finds the row number holding the smallest value.
softest_row_number = predictions.K_VRH_pred.idxmin()
softest = predictions.loc[softest_row_number]

print(f"The softest crystal is {softest.formula}, with K = {softest.K_VRH_pred:.2f}")
print(f"Its give-or-take is {softest.K_VRH_spread_log10:.4f} in log units - quite large.")
print()

# Make 20,000 guesses THE RIGHT WAY (in log units, then convert back).
right_way = 10 ** (np.log10(softest.K_VRH_pred)
                   + softest.K_VRH_spread_log10 * dice.standard_normal(20000))

# Make 20,000 guesses THE WRONG WAY (adding the wobble straight onto the modulus).
# math.log(10) * spread * K converts the log give-or-take into a rough
# equivalent in ordinary units, so the comparison is fair.
equivalent_spread = math.log(10) * softest.K_VRH_spread_log10 * softest.K_VRH_pred
wrong_way = softest.K_VRH_pred + equivalent_spread * dice.standard_normal(20000)

# Count how many guesses came out at zero or below.
# (wrong_way <= 0) gives a true/false for each guess; .sum() counts the trues.
print(f"Working in logs      -> smallest guess {right_way.min():8.3f}, "
      f"impossible values: {(right_way <= 0).sum()}")
print(f"Adding directly      -> smallest guess {wrong_way.min():8.3f}, "
      f"impossible values: {(wrong_way <= 0).sum()}")
print()
print("The wrong way produces negative moduli, which are physically meaningless")
print("and would crash the square root. The log way cannot, ever.")

---
# Part 5 · Why 2000 guesses?

Why not 50? Why not a million?

More guesses give a steadier answer, but with diminishing returns — to halve the wobble in
your answer you need **four times** as many guesses. So there is a point where more guessing
stops being worth it.

Let us find that point by running the same crystal with different numbers of guesses.

**More Python:**

- A `for` loop repeats the same block of code once for each item in a list.
- `for n in [50, 100, 250]:` means *"do the following three times: once with n meaning 50,
  once with n meaning 100, once with n meaning 250"*.
- The **indented** lines below the `for` are the ones that get repeated.

In [ ]:
# The different numbers of guesses we want to try.
sizes_to_try = [50, 100, 500, 2000, 20000]

print(f"{'guesses':>10}   {'low end':>9}   {'high end':>9}")

# Repeat everything indented below, once for each size in the list.
for n in sizes_to_try:
    # Restart the random machine with the same seed each time, so the ONLY
    # thing changing between runs is how many guesses we make.
    d = np.random.RandomState(0)

    # Make n guesses for K and n for G, exactly as before.
    k = 10 ** (np.log10(K_guess) + K_giveTake * d.standard_normal(n))
    g = 10 ** (np.log10(G_guess) + G_giveTake * d.standard_normal(n))

    # Run the physics and take the two percentiles.
    result = slack_physics(k, g, volume, density, mass, n_atoms)["kappa_cal"]
    lo, hi = np.percentile(result, [5, 95])

    # Print one line of the table.
    print(f"{n:>10,}   {lo:>9.4f}   {hi:>9.4f}")

print()
print("With only 50 guesses the answer jumps around. By 2000 it has settled -")
print("the remaining wiggle is far smaller than the width of the interval itself,")
print("so guessing more would be polishing a number that is already good enough.")

---
# Part 6 · The catch — and this is the important part

Everything above is arithmetically correct. And the answer it gives is still **too
confident**.

Here is how we know. We tested it on crystals where we already knew the true answer, and
asked a simple question:

> *When the method said "I'm 90% sure the truth is in this range" — how often was it?*

If the method were honest, the answer would be 90%.

In [ ]:
# Open the results of that test.
check = pd.read_csv(os.path.join(PROJECT, "results", "cgcnn", "calibration_check.csv"))

# Find the row for the 90% claim.
row_90 = check[check.nominal == 0.9].iloc[0]

print("When the method claimed to be 90% sure:")
print(f"   for K, the truth was actually inside the range {100 * row_90.K_VRH_raw:.1f}% of the time")
print(f"   for G, the truth was actually inside the range {100 * row_90.G_VRH_raw:.1f}% of the time")
print()
print("It should have been 90%. It was about half that.")
print("The method is roughly HALF as confident as it claims to be.")

### Why is it wrong?

Remember where the give-or-take came from: **how much our three machines disagreed**.

But three machines built the same way, trained on the same data, can be **wrong together**.
When they are, they *agree* with each other — so the disagreement is small, so the
give-or-take is small, and the method reports high confidence in an answer that is wrong.

An everyday version: ask three friends who all read the same newspaper what will happen in
the election. They will agree closely. That agreement tells you nothing about whether they
are right — only that they share a source.

So:

> **Disagreement between models measures only part of the error.**
> **It misses whatever they all get wrong together.**

Here, that missing part is large: the true error is about **4 to 6 times** bigger than the
disagreement suggests.

### What to do about it

Two honest options:

1. **Widen the interval** by a factor measured against known answers. Doing that restores
   the honest 90%.
2. **Report it as-is, but say plainly what it is** — a measure of model disagreement, not a
   full confidence interval.

**The general lesson, worth taking away even if you forget everything else here:** an
ensemble's spread is a *floor* on its uncertainty, never the whole of it. Check it against
known answers before believing it.

---
# Part 7 · Proof that this notebook matches the real pipeline

Everything above was written out step by step for teaching. The real pipeline does it in a
single function call.

If those two disagreed, this notebook would be explaining something other than what
actually runs. So let us check rather than assume — we run both and compare every number.

In [ ]:
# Load the real pipeline's own Monte Carlo function.
real_function = physics_file.monte_carlo_kappa

# Take 200 crystals. This table already holds every column the function needs,
# including the give-or-take columns, so nothing has to be joined on.
# .copy() makes a private copy so we cannot accidentally alter the original.
small = facts_table.head(200).copy()

# Run the REAL function.
theirs = real_function(small, n_samples=2000, seed=0)

# Now the same thing our way, for all 200 crystals at once.
d = np.random.RandomState(0)
rows = len(small)
# [:, None] turns a flat list of 200 numbers into a single column, so that each
# crystal gets its own guesses rather than sharing one set.
wk = d.standard_normal((rows, 2000))
wg = d.standard_normal((rows, 2000))
lk = np.log10(small["K_VRH_pred"].values)[:, None] + small["K_VRH_spread_log10"].values[:, None] * wk
lg = np.log10(small["G_VRH_pred"].values)[:, None] + small["G_VRH_spread_log10"].values[:, None] * wg
column = lambda name: small[name].values[:, None]
ours = slack_physics(10 ** lk, 10 ** lg, column("Volume (A3)"),
                     column("Density (g cm-3)"), column("Atomic mass (amu)"),
                     column("Number of Atoms"))["kappa_cal"]

# Take the same percentiles, one per crystal (axis=1 means "across the guesses").
our_low  = np.nanpercentile(ours, 5,  axis=1)
our_high = np.nanpercentile(ours, 95, axis=1)

# Compare. abs(...) removes minus signs; .max() finds the worst disagreement.
worst_low  = np.abs(our_low  - theirs["Kappa_cal_p05"]).max()
worst_high = np.abs(our_high - theirs["Kappa_cal_p95"]).max()

print(f"Biggest disagreement on the low end  across 200 crystals: {worst_low:.3e}")
print(f"Biggest disagreement on the high end across 200 crystals: {worst_high:.3e}")
print()
if max(worst_low, worst_high) == 0:
    print("EXACTLY ZERO. This notebook does precisely what the pipeline does.")
else:
    print("They differ - so this notebook is NOT explaining what really runs.")

---
# The whole thing in six sentences

1. Our machine predicts K and G, but only roughly — it gives a best guess and a give-or-take.
2. To find out what that means for κ, we make **2000 random guesses** of K and G that
   respect the give-or-take.
3. We run the **real physics** on all 2000, giving 2000 possible values of κ.
4. We report the range that holds the **middle 90%** of them.
5. We work in **logarithms** throughout, because the error is a percentage and because it
   makes an impossible negative modulus impossible.
6. **The resulting range is about half as wide as it should be**, because model
   disagreement misses whatever the models get wrong together — so widen it, or say plainly
   what it measures.